# Centaur IME: 韓国語の並べ替えを速くする（候補の数と模型の大きさ）

今の模型（rr-ko.pt）で、候補を上位 2・4・8・16 個にしたときの点を測り、小さい模型（d 256・4 層、d 192・4 層）を学習して
同じ試験で比べる。速さは GitHub の CPU で別に測る（tools/models/speed_bench.py）。

使い方: L4 GPU を選び、「すべてのセルを実行」。最初のセルで Google ドライブにつなぐ（許可する）。


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/CentaurIME/models/2026-11-07'
import os; print('保存先', OUT, os.path.exists(OUT + '/rr-ko.pt'))

In [ ]:
!rm -rf repo && git clone -q --depth 1 -b work/lang-measure https://github.com/J-KAMMA/centaur-ime.git repo && git -C repo log -1 --format='%H %s'
!pip -q install pyarrow==21.0.0

In [ ]:
!python repo/tools/fetch_source.py tatoeba-kor data/ko
!python repo/tools/fetch_source.py wiki-ko data/ko
!python repo/tools/fetch_source.py cc100-ko data/ko --lines 300000

In [ ]:
!python repo/tools/ko_ime/measure.py --tatoeba data/ko/kor_sentences.tsv.bz2 --wiki $(ls data/ko/*.parquet | head -1) --out m --nbest-out nbest.jsonl
!wc -l nbest.jsonl

In [ ]:
# 今の模型（d 384・8 層）で、候補の数ごとの点
!python repo/tools/models/train_task.py --task rr-ko --tatoeba data/ko/kor_sentences.tsv.bz2 --wiki $(ls data/ko/*.parquet | head -1) --cc100 data/ko/cc100-ko.300000.txt --extra 300000 --batch 256 --amp --nbest-file nbest.jsonl --out "$OUT" --name rr-ko-topk --load "$OUT/rr-ko.pt"
print(open(OUT + '/rr-ko-topk-result.txt', encoding='utf-8').read())

In [ ]:
# 小さい模型 1: d 256・4 層（計算はおよそ 4 分の 1）
!python repo/tools/models/train_task.py --task rr-ko --tatoeba data/ko/kor_sentences.tsv.bz2 --wiki $(ls data/ko/*.parquet | head -1) --cc100 data/ko/cc100-ko.300000.txt --extra 300000 --batch 256 --amp --nbest-file nbest.jsonl --d 256 --layers 4 --heads 4 --minutes 20 --out "$OUT" --name rr-ko-d256
print(open(OUT + '/rr-ko-d256-result.txt', encoding='utf-8').read())

In [ ]:
# 小さい模型 2: d 192・4 層
!python repo/tools/models/train_task.py --task rr-ko --tatoeba data/ko/kor_sentences.tsv.bz2 --wiki $(ls data/ko/*.parquet | head -1) --cc100 data/ko/cc100-ko.300000.txt --extra 300000 --batch 256 --amp --nbest-file nbest.jsonl --d 192 --layers 4 --heads 4 --minutes 20 --out "$OUT" --name rr-ko-d192
print(open(OUT + '/rr-ko-d192-result.txt', encoding='utf-8').read())

In [ ]:
from google.colab import runtime
runtime.unassign()